# 05 — Export Research Model Artifact

In [1]:
import sys, json, joblib
from pathlib import Path
from sklearn.preprocessing import LabelEncoder
from sklearn.base import clone
NB_DIR = Path.cwd()
if not (NB_DIR / 'notebook_utils.py').exists():
    NB_DIR = next(p for p in [NB_DIR, *NB_DIR.parents] if (p / 'src' / 'notebooks' / 'notebook_utils.py').exists()) / 'src' / 'notebooks'
sys.path.insert(0, str(NB_DIR))
from notebook_utils import *

Run notebook 03 first so the selected model exists in `outputs/ml_research/`.

In [2]:
raw_df, chosen = select_training_table()
df = standardize_training_table(raw_df)
X, y_text, FEATURE_NAMES = build_feature_table(df)
le = LabelEncoder()
y = le.fit_transform(y_text)
selected = json.load(open(PROCESSED_DIR / 'selected_model.json', encoding='utf-8'))['best_model']
candidate = joblib.load(PROCESSED_DIR / f'model_{safe_model_filename(selected)}.joblib')
final_model = clone(candidate).fit(X, y)
print('Final model:', selected)
print('Training rows:', len(X))

Final model: KNN
Training rows: 526


In [3]:
bundle = {
    'model': final_model,
    'label_encoder': le,
    'feature_names': FEATURE_NAMES,
    'model_name': selected,
    'training_rows': int(len(X)),
    'classes': list(le.classes_),
    'input_schema_version': 'EOR_ATLAS_ML_V2',
    'primary_features': FEATURE_NAMES,
    'fuzzy_used_as_model_feature': False,
    'trained_from': chosen,
    'notes': [
        'Research model only; deterministic Excel screening remains independent.',
        'Fuzzy suitability is calculated separately at inference time.',
        'Validate generalization with field/project-grouped data when IDs become available.'
    ],
}
bundle_path = ARTIFACT_DIR / 'eor_research_best.joblib'
joblib.dump(bundle, bundle_path)
metadata = {k:v for k,v in bundle.items() if k not in {'model','label_encoder'}}
json.dump(metadata, open(ARTIFACT_DIR / 'eor_research_metadata.json','w',encoding='utf-8'), indent=2, default=str)
print('Exported:', bundle_path)

Exported: c:\Users\mnabielizzuddin.radz\OneDrive - PETRONAS\Reservoir Engineering\Programming_Python_Projects\EOR ATLAS\EORWEB\EORWEBDEV\outputs\model_artifacts\eor_research_best.joblib


In [4]:
loaded = joblib.load(bundle_path)
print('Artifact smoke test passed')
print('Model:', loaded['model_name'])
print('Classes:', loaded['classes'])
print('Features:', len(loaded['feature_names']))

Artifact smoke test passed
Model: KNN
Classes: ['Combustion', 'HC immiscible', 'Hot water', 'Miscible CO2', 'Miscible HC', 'Miscible acid gas', 'Nitrogen immiscible', 'Polymer', 'Steam']
Features: 17
